In [ ]:
import QuantLib as ql
import torch
import torch.nn as nn
import time
import numpy as np
from scipy.optimize import minimize
import torch.optim as optim
from functools import reduce

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'  # Use GPU if available

In [ ]:
print(device)

## Deep Importance Sampling
This notebook uses deep importance sampling and Monte Carlo to price an arithmetic Asian option. The pricing accuracy and execution time is compared with quantlib and a PyTorch simple Monte Carlo simulation.

### QuantLib Asian

In [ ]:
def value_asian_option(S, K, r, sigma, T, option_type, n):
    """
    Value an Asian option using QuantLib with a Black-Scholes model.
    
    Parameters:
    S (float): Initial stock price
    K (float): Strike price
    r (float): Risk-free rate
    sigma (float): Volatility
    T (float): Time to maturity (in years)
    option_type (str): 'call' or 'put'
    n (int): Number of time steps
    
    Returns:
    float: Option price
    """
    
    # Set up the QuantLib environment
    calendar = ql.NullCalendar()
    day_count = ql.Actual365Fixed()
    
    # Set up the dates
    maturity_date = ql.Date().todaysDate() + int(T * 365)
    settlement_date = ql.Date().todaysDate()
    fixing_dates = [settlement_date + int(i * T * 365 / n) for i in range(1, n + 1)]
    print(len(fixing_dates))
    
    # Set up the option payoff and exercise type
    payoff = ql.PlainVanillaPayoff(ql.Option.Call if option_type == 'call' else ql.Option.Put, K)
    exercise = ql.EuropeanExercise(maturity_date)
    pastFixings = 0
    aRAccumulator = 0.0
    
    option = ql.DiscreteAveragingAsianOption(
        ql.Average.Arithmetic, aRAccumulator, pastFixings, fixing_dates, payoff, exercise
    )
    
    # Set up the Black-Scholes process
    spot_handle = ql.QuoteHandle(ql.SimpleQuote(S))
    div_ts = ql.YieldTermStructureHandle(ql.FlatForward(settlement_date, 0.0, day_count))
    flat_ts = ql.YieldTermStructureHandle(ql.FlatForward(settlement_date, r, day_count))
    vol_ts = ql.BlackVolTermStructureHandle(ql.BlackConstantVol(settlement_date, calendar, sigma, day_count))
    bs_process = ql.BlackScholesMertonProcess(spot_handle, div_ts, flat_ts, vol_ts)
    
    # Set up the engine and price the option
    engine = ql.FdBlackScholesAsianEngine(bs_process, tGrid=100, xGrid=100, aGrid=50)
    option.setPricingEngine(engine)
    
    return option.NPV()

In [ ]:
S = 1 # Initial stock price
r = 0.05   # Risk-free rate
sigma = 0.2  # Volatility
T = 1      # Time to maturity (in years)
option_type = 'call'  # Option type ('call' or 'put')
n = 12     # Number of time steps
F = S * np.exp(T * r)
K = 1.3 * F    # Strike price

start_time = time.time()
price = value_asian_option(S, K, r, sigma, T, option_type, n)
total_time = time.time() - start_time
print(f"Asian option price: {price}")

In [ ]:
print(f"PDE Execution Time: {total_time} seconds")

### PyTorch Monte Carlo

In [ ]:
def simulate_asset_paths(S0, r, sigma, T, dt, N, M, mu_shift, device='cpu'):
    # Initialize the tensor to store paths on the specified device
    paths = torch.zeros((M, N + 1), device=device)
    weights = torch.ones((M), device=device)
    
    # Set the initial asset price
    paths[:, 0] = S0
    
    # Precompute constants
    drift = (r - 0.5 * sigma ** 2) * dt
    sqrdt = torch.sqrt(torch.tensor(dt, device=device))
    diffusion = sigma * sqrdt
    
    # Importance sampling shift
    drift_IS = (r - 0.5 * sigma ** 2 - mu_shift * sigma) * dt
    weight_factor = np.exp(-mu_shift**2 * dt / 2.0)
    
    # Generate paths
    for t in range(1, N + 1):
        Z = torch.randn(M, device=device)
        paths[:, t] = paths[:, t - 1] * torch.exp(drift_IS + diffusion * Z)
        weights = weights * weight_factor * torch.exp(mu_shift * sqrdt * Z)
    
    return paths, weights

In [ ]:
def value_asian_option_pytorch(S, K, r, sigma, T, option_type, n, num_paths, 
                                                   seed, mu_shift, device='cpu'):
    """
    Value an arithmetic Asian option using PyTorch with Monte Carlo simulation and importance sampling.
    
    Parameters:
    S (float): Initial stock price
    K (float): Strike price
    r (float): Risk-free rate
    sigma (float): Volatility
    T (float): Time to maturity (in years)
    option_type (str): 'call' or 'put'
    n (int): Number of time steps
    num_paths (int): Number of Monte Carlo paths
    seed (int): Random seed for reproducibility
    mu_shift (float): Importance sampling drift shift
    device (str): Device to run the simulation on ('cpu' or 'cuda')
    
    Returns:
    float: Option price
    """
    torch.manual_seed(seed)
    device = torch.device(device)
    
    # Time step size
    dt = T / n
    
    # Generate random paths with importance sampling
    paths, weights = simulate_asset_paths(S, r, sigma, T, dt, n, num_paths, mu_shift, device=device)
    
    # Calculate the average price
    average_price = paths[:, 1:].mean(dim=1)
    
    # Calculate payoff
    if option_type == 'call':
        payoffs = torch.maximum(average_price - K, torch.tensor(0.0, device=device))
    else:
        payoffs = torch.maximum(K - average_price, torch.tensor(0.0, device=device))
    
    # Discount payoffs
    discounted_payoffs = np.exp(-r * T) * payoffs
    
    # Apply importance sampling weight factor
    weighted_payoffs = discounted_payoffs * weights
    
    # Option price
    option_price = weighted_payoffs.mean().item()
    return option_price

In [ ]:
def estimate_standard_error(S, K, r, sigma, T, option_type, n, num_paths, 
                                                num_runs, mu_shift, device='cpu'):
    """
    Estimate the standard error using multiple runs with different seeds and importance sampling.
    
    Parameters:
    S (float): Initial stock price
    K (float): Strike price
    r (float): Risk-free rate
    sigma (float): Volatility
    T (float): Time to maturity (in years)
    option_type (str): 'call' or 'put'
    n (int): Number of time steps
    num_paths (int): Number of Monte Carlo paths
    num_runs (int): Number of runs with different seeds
    mu_shift (float): Importance sampling drift shift
    device (str): Device to run the simulation on ('cpu' or 'cuda')
    
    Returns:
    float: Mean option price
    float: Standard error
    float: Total time taken
    """
    prices = []
    start_time = time.time()
    
    for seed in range(num_runs):
        price = value_asian_option_pytorch(S, K, r, sigma, T, option_type, n, num_paths, seed, mu_shift, device)
        prices.append(price)
    
    total_time = time.time() - start_time
    mean_price = sum(prices) / num_runs
    std_error = np.std(prices) / np.sqrt(num_runs)
    
    return mean_price, std_error, total_time

In [ ]:
num_paths = 100000  # Number of Monte Carlo paths
num_runs = 10      # Number of runs with different seed
mu_shift = 0.0

In [ ]:
mean_price, mean_error, total_time = estimate_standard_error(S, K, r, sigma, T, option_type, 
                                            n, num_paths, num_runs, mu_shift, device)
print(f"Mean option price: {mean_price}")
print(f"Standard error: {mean_error}")
print(f"Total time taken: {total_time} seconds")

In [ ]:
def optimize_mu_shift(S, K, r, sigma, T, option_type, n, num_paths, num_runs, device='cpu'):
    """
    Optimize the choice of mu_shift to minimize the standard error.
    
    Parameters:
    S (float): Initial stock price
    K (float): Strike price
    r (float): Risk-free rate
    sigma (float): Volatility
    T (float): Time to maturity (in years)
    option_type (str): 'call' or 'put'
    n (int): Number of time steps
    num_paths (int): Number of Monte Carlo paths
    num_runs (int): Number of runs with different seeds
    device (str): Device to run the simulation on ('cpu' or 'cuda')
    
    Returns:
    float: Optimal mu_shift
    float: Corresponding standard error
    """
    def objective(mu_shift):
        _, std_error, _ = estimate_standard_error(S, K, r, sigma, T, option_type, 
                                                  n, num_paths, num_runs, mu_shift.item(), device)
        return std_error
    
    result = minimize(objective, x0=0.0, bounds=[(-1.0, 1.0)], method='L-BFGS-B')
    optimal_mu_shift = result.x[0]
    optimal_std_error = result.fun
    return optimal_mu_shift, optimal_std_error

In [ ]:
# Optimize mu_shift
optimal_mu_shift, optimal_std_error = optimize_mu_shift(S, K, r, sigma, T, option_type, n, 
                                                        num_paths, num_runs, device)
print(f"Optimal mu_shift: {optimal_mu_shift}")
print(f"Corresponding standard error: {optimal_std_error}")

In [ ]:
mean_price, mean_error1, total_time = estimate_standard_error(S, K, r, sigma, T, option_type, 
                                            n, num_paths, num_runs, optimal_mu_shift, device)

In [ ]:
print(f"Mean option price: {mean_price}")
print(f"Standard error: {mean_error1}")
print(f"Total time taken: {total_time} seconds")

### Deep Importance Sampling

In [ ]:
class MuShiftNetwork(nn.Module):
    def __init__(self, input_size):
        super(MuShiftNetwork, self).__init__()
        self.fc1 = nn.Linear(input_size, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1)
        self.relu = nn.ReLU()
        
    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.relu(out)
        out = self.fc3(out)
        return out

In [ ]:
def simulate_asset_paths_nn(S0, r, sigma, T, dt, N, M, mu_models, device='cpu'):
    # Initialize the tensor to store paths on the specified device
    paths = [torch.zeros((M, 1), device=device) for i in range(N+1)]
    weights_lst = list()
    
    # Set the initial asset price
    paths[0][:,0] = S0
    
    # Precompute constants
    drift = (r - 0.5 * sigma ** 2) * dt
    sqrdt = torch.sqrt(torch.tensor(dt, device=device))
    diffusion = sigma * sqrdt
    
    # Generate paths
    for t in range(1, N + 1):
        mu_model = mu_models[t-1]
        Z = torch.randn(M, device=device)
        concat_paths = torch.cat(paths[:t], dim=1)
        mu_shift = mu_model(concat_paths).squeeze()
        
        drift_IS = (r - 0.5 * sigma ** 2 - mu_shift * sigma) * dt
        weight_factor = torch.exp(-mu_shift**2 * dt / 2.0)
        paths[t][:,0] = paths[t-1][:, 0] * torch.exp(drift_IS + diffusion * Z)
        weights_lst.append(weight_factor * torch.exp(mu_shift * sqrdt * Z))
    
    return torch.cat(paths, dim=1), reduce(torch.mul, weights_lst)

In [ ]:
def value_asian_option_pytorch_nn(S, K, r, sigma, T, option_type, n, num_paths, 
                               seed, mu_models, device='cpu'):
    """
    Value an arithmetic Asian option using PyTorch with Monte Carlo simulation and importance sampling.
    
    Parameters:
    S (float): Initial stock price
    K (float): Strike price
    r (float): Risk-free rate
    sigma (float): Volatility
    T (float): Time to maturity (in years)
    option_type (str): 'call' or 'put'
    n (int): Number of time steps
    num_paths (int): Number of Monte Carlo paths
    seed (int): Random seed for reproducibility
    mu_model (nn.Module): Neural network model to predict mu_shift
    device (str): Device to run the simulation on ('cpu' or 'cuda')
    
    Returns:
    float: Option price
    """
    torch.manual_seed(seed)
    device = torch.device(device)
    
    # Time step size
    dt = T / n
    
    # Generate random paths with importance sampling
    paths, weights = simulate_asset_paths_nn(S, r, sigma, T, dt, n, num_paths, mu_models, device=device)
    
    # Calculate the average price
    average_price = paths[:, 1:].mean(dim=1)
    
    # Calculate payoff
    if option_type == 'call':
        payoffs = torch.maximum(average_price - K, torch.tensor(0.0, device=device))
    else:
        payoffs = torch.maximum(K - average_price, torch.tensor(0.0, device=device))
    
    # Discount payoffs
    discounted_payoffs = np.exp(-r * T) * payoffs
    
    # Apply importance sampling weight factor
    weighted_payoffs = discounted_payoffs * weights
    
    # Option price
    option_price = weighted_payoffs.mean().item()
    return option_price

In [ ]:
def estimate_standard_error_nn(S, K, r, sigma, T, option_type, n, num_paths, 
                            num_runs, mu_models, device='cpu'):
    """
    Estimate the standard error using multiple runs with different seeds and importance sampling.
    
    Parameters:
    S (float): Initial stock price
    K (float): Strike price
    r (float): Risk-free rate
    sigma (float): Volatility
    T (float): Time to maturity (in years)
    option_type (str): 'call' or 'put'
    n (int): Number of time steps
    num_paths (int): Number of Monte Carlo paths
    num_runs (int): Number of runs with different seeds
    mu_model (nn.Module): Neural network model to predict mu_shift
    device (str): Device to run the simulation on ('cpu' or 'cuda')
    
    Returns:
    float: Mean option price
    float: Standard error
    float: Total time taken
    """
    prices = []
    start_time = time.time()
    
    for seed in range(num_runs):
        price = value_asian_option_pytorch_nn(S, K, r, sigma, T, option_type, n, num_paths, seed, mu_models, device)
        prices.append(price)
    
    total_time = time.time() - start_time
    mean_price = sum(prices) / num_runs
    std_error = np.std(prices) / np.sqrt(num_runs)
    
    return mean_price, std_error, total_time

In [ ]:
mu_models = list()
for i in range(1, n+1):
    mu_model = MuShiftNetwork(i).to(device)
    mu_models.append(mu_model)

# Estimate the standard error
mean_price, mean_error2, total_time = estimate_standard_error_nn(S, K, r, sigma, T, option_type, 
                                                             n, num_paths, num_runs, mu_models, device)
print(f"Mean option price: {mean_price}")
print(f"Standard error: {mean_error2}")
print(f"Total time taken: {total_time} seconds")

In [ ]:
def train_mu_models(S, K, r, sigma, T, option_type, n, num_paths, 
                    num_epochs, learning_rate, device='cpu', lmda = 0.00):
    mu_models = [MuShiftNetwork(i).to(device) for i in range(1, n + 1)]
    params = list()
    for model in mu_models:
        params += list(model.parameters())
    optimizer = optim.Adam(params, lr=learning_rate)
    
    for epoch in range(num_epochs):
        total_loss = 0.0
        for model in mu_models:
            model.train()
        optimizer.zero_grad()
            
        # Generate random paths
        paths, weights = simulate_asset_paths_nn(S, r, sigma, T, T/n, n, num_paths, mu_models, device=device)
            
        # Calculate the average price
        average_price = paths[:, 1:].mean(dim=1)
            
            # Calculate payoff
        if option_type == 'call':
            payoffs = torch.maximum(average_price - K, torch.tensor(0.0, device=device))
        else:
            payoffs = torch.maximum(K - average_price, torch.tensor(0.0, device=device))
            
        # Discount payoffs
        discounted_payoffs = np.exp(-r * T) * payoffs
            
        # Apply importance sampling weight factor
        weighted_payoffs = discounted_payoffs * weights
            
        # Calculate the variance of weighted_payoffs
        variance_loss = weighted_payoffs.var()
            
        # Regularization term to ensure weights remain close to 1.0
        weight_reg = ((weights - 1.0) ** 2).mean()
            
        # Total loss
        loss = variance_loss + lmda * weight_reg
        loss.backward()
        optimizer.step()
            
        total_loss += loss.item()
        
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {total_loss/n}")
    
    return mu_models

In [ ]:
num_epochs = 100    # Number of epochs for training
learning_rate = 0.001 # Learning rate

In [ ]:
start_time = time.time()
mu_models = train_mu_models(S, K, r, sigma, T, option_type, n, 
                            num_paths, num_epochs, learning_rate, device)
total_time = time.time() - start_time

In [ ]:
print(f"Training time: {total_time} seconds")

In [ ]:
# Estimate the standard error
mean_price, mean_error3, total_time = estimate_standard_error_nn(S, K, r, sigma, T, option_type, 
                                                             n, num_paths, num_runs, mu_models, device)
print(f"Mean option price: {mean_price}")
print(f"Standard error: {mean_error3}")
print(f"Total time taken: {total_time} seconds")

In [ ]:
fac = mean_error3 / mean_error1
fac

In [ ]:
num_paths = 10000000  # Number of Monte Carlo paths
num_runs = 10      # Number of runs with different seed
mu_shift = 0.0

mean_price, mean_error, total_time = estimate_standard_error(S, K, r, sigma, T, option_type, 
                                            n, num_paths, num_runs, mu_shift, device)
print(f"Mean option price: {mean_price}")
print(f"Standard error: {mean_error}")
print(f"Total time taken: {total_time} seconds")